# Evaluation of the effect of RAG and Chain-of-Thought on the performance of large language models

* Load the packages and the MMLU datasets

In [4]:
#This notebook will evaluate different answer generation functions on the mmlu dataset, summarizing the results and outputting them to CSV files.

import pandas as pd
import import_datasets
import general_utils
from generate_answer_cot import generate_answer_cot
#Import other answer generation functions as needed
from evaluate_model_fast import evaluate_model_fast
from rag import rag_pipeline
from generate_answer_rag import generate_answer_rag
from generate_answer_std import generate_answer_std

client = general_utils.load_gemini_client()



In [3]:
#Generate 100 samples from the mmlu datasets
dataset_names = ['anatomy', 'formal_logic', 'moral_scenarios']
datasets = import_datasets.import_datasets(dataset_names)
df_anatomy = datasets['anatomy']
df_formal_logic = datasets['formal_logic']
df_moral_scenarios = datasets['moral_scenarios']

## 1. Evaluate Standard Model Performance

We simply prompted Gemini to answer the questions from the MMLU datasets. The prompt we provided is very basic prompt with no prompt engineering or chain-of-thought prompting.

In [ ]:
#Evaluate the accuracy of each model on each dataset. Store the results in dataframes.
#Evaluate the standard model performance on each dataset
accuracy_anatomy_std, results_df_anatomy_std = evaluate_model_fast(df_anatomy, generate_answer_std, client=client)
accuracy_formal_logic_std, results_df_formal_logic_std = evaluate_model_fast(df_formal_logic, generate_answer_std, client=client)
accuracy_moral_scenarios_std, results_df_moral_scenarios_std = evaluate_model_fast(df_moral_scenarios, generate_answer_std, client=client)

# save as csv
general_utils.save_df_to_csv(results_df_anatomy_std, "results_df_anatomy_std.csv")
general_utils.save_df_to_csv(results_df_formal_logic_std, "results_df_formal_logic_std.csv")
general_utils.save_df_to_csv(results_df_moral_scenarios_std, "results_df_moral_scenarios_std.csv")


## 2. Evaluate Chain-of-Thought Model Performance

We prompted Gemini to answer the questions from the MMLU datasets. The prompt we provided is asking the model to think step-by-step before answering the question.

In [16]:
#Evaluate the cot model performance on each dataset
accuracy_anatomy_cot, results_df_anatomy_cot = evaluate_model_fast(df_anatomy, generate_answer_cot, client=client)
accuracy_formal_logic_cot, results_df_formal_logic_cot = evaluate_model_fast(df_formal_logic, generate_answer_cot, client=client)
accuracy_moral_scenarios_cot, results_df_moral_scenarios_cot = evaluate_model_fast(df_moral_scenarios, generate_answer_cot, client=client)

# save as csv
general_utils.save_df_to_csv(results_df_anatomy_cot, "results_df_anatomy_cot.csv")
general_utils.save_df_to_csv(results_df_formal_logic_cot, "results_df_formal_logic_cot.csv")
general_utils.save_df_to_csv(results_df_moral_scenarios_cot, "results_df_moral_scenarios_cot.csv")

## 3. Evaluate Model Performance with RAG
> Warning: Takes ~50 minutes to run!
* Convert concept net to RAG vector store

In [2]:
# build the rag pipeline to vectorize concept net
file_path = "../input_data/concept_net.txt"
vector_store = rag_pipeline(file_path, split_by="newline", chunk_size=1000, chunk_overlap=0, add_start_index=True, batch_processing_size=700)

Total splits: 130923
Processing 0 to 700
Processing 700 to 1400
Processing 1400 to 2100
Processing 2100 to 2800
Processing 2800 to 3500
Rate limit hit, waiting 20 seconds before retrying...
Processing 3500 to 4200
Processing 4200 to 4900
Processing 4900 to 5600
Processing 5600 to 6300
Processing 6300 to 7000
Processing 7000 to 7700
Processing 7700 to 8400
Rate limit hit, waiting 20 seconds before retrying...
Processing 8400 to 9100
Processing 9100 to 9800
Processing 9800 to 10500
Processing 10500 to 11200
Rate limit hit, waiting 20 seconds before retrying...
Processing 11200 to 11900
Processing 11900 to 12600
Processing 12600 to 13300
Processing 13300 to 14000
Processing 14000 to 14700
Processing 14700 to 15400
Processing 15400 to 16100
Processing 16100 to 16800
Rate limit hit, waiting 20 seconds before retrying...
Processing 16800 to 17500
Processing 17500 to 18200
Processing 18200 to 18900
Processing 18900 to 19600
Rate limit hit, waiting 20 seconds before retrying...
Processing 1960

In [7]:
import pandas as pd
from concurrent.futures import ThreadPoolExecutor

def process_row(row, answer_generation_function, client, vector_store, k):
    """Helper to process a single row for parallel execution."""
    question = row['question']
    choices = row['choices'][:4]  # Simplified slicing
    correct_choice = {0: 'A', 1: 'B', 2: 'C', 3: 'D'}[row['answer']]
    subject = row.get('subject', 'N/A')
    k_to_store = None
    
    kwargs = {"client": client}
    if "rag" in answer_generation_function.__name__.lower():
        kwargs.update({"vector_store": vector_store, "k": k})
        k_to_store = k

    # Retry logic
    for i in range(5):
        try:
            model_answer = answer_generation_function(question, choices, **kwargs)
            break # Success! Exit the loop
        except Exception as e:
            # Captures the specific error message as the reasoning
            error_msg = f"Attempt {i+1} failed: {str(e)}"
            model_answer = (f"Error", error_msg)
            
    return {
        'question': question,
        'model_answer': model_answer[0],
        'model_reasoning': model_answer[1],
        'correct_answer': correct_choice,
        'correct': model_answer[0] == correct_choice,
        'subject': subject,
        'k': k_to_store
    }

def evaluate_model_fast(df, answer_generation_function, client, debugging=False, vector_store=None, k=1, max_workers=10):
    # Convert DF to list of dicts for faster iteration
    rows = df.to_dict('records')
    
    # Use ThreadPoolExecutor to call the API in parallel
    results = []
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        results = list(executor.map(lambda r: process_row(r, answer_generation_function, client, vector_store, k), rows))

    results_df = pd.DataFrame(results)

    # --- New Filtering Logic ---
    # Filter for rows where the model actually gave a choice (A, B, C, or D)
    # and didn't return "Error"
    valid_results = results_df[results_df['model_answer'] != "Error"]
    
    if not valid_results.empty:
        accuracy = valid_results['correct'].mean()
    else:
        accuracy = 0.0  # Handle case where everything failed
    # ---------------------------
    
    if debugging:
        error_count = len(results_df) - len(valid_results)
        print(f"Evaluation complete.")
        print(f"Total Rows: {len(results_df)} | Errors Excluded: {error_count}")
        print(f"Adjusted Accuracy: {accuracy:.2%}")
        
    return accuracy, results_df

In [ ]:
accuracy_anatomy_rag, results_df_anatomy_rag = evaluate_model_fast(df_anatomy, generate_answer_rag, client=client, vector_store=vector_store, k=1)
accuracy_formal_logic_rag, results_df_formal_logic_rag = evaluate_model_fast(df_formal_logic, generate_answer_rag, client=client, vector_store=vector_store, k=1)
accuracy_moral_scenarios_rag, results_df_moral_scenarios_rag = evaluate_model_fast(df_moral_scenarios, generate_answer_rag, client=client, vector_store=vector_store, k=1)

# save as csv
general_utils.save_df_to_csv(results_df_anatomy_rag, "results_df_anatomy_rag.csv")
general_utils.save_df_to_csv(results_df_formal_logic_rag, "results_df_formal_logic_rag.csv")
general_utils.save_df_to_csv(results_df_moral_scenarios_rag, "results_df_moral_scenarios_rag.csv")

In [18]:
#Combine results for each model into dataframes:
results_df_std = pd.concat([results_df_anatomy_std, results_df_formal_logic_std, results_df_moral_scenarios_std], ignore_index=True)
results_df_cot = pd.concat([results_df_anatomy_cot, results_df_formal_logic_cot, results_df_moral_scenarios_cot], ignore_index=True)
results_df_rag = pd.concat([results_df_anatomy_rag, results_df_formal_logic_rag, results_df_moral_scenarios_rag], ignore_index=True)

In [19]:
#create a dataframe to summarize the results
results_summary = pd.DataFrame({
    'Dataset': ['Standard', 'CoT', 'RAG'],
    'Anatomy': [accuracy_anatomy_std, accuracy_anatomy_cot, accuracy_anatomy_rag],
    'Formal Logic': [accuracy_formal_logic_std, accuracy_formal_logic_cot, accuracy_formal_logic_rag],
    'Moral Scenarios': [accuracy_moral_scenarios_std, accuracy_moral_scenarios_cot, accuracy_moral_scenarios_rag]
}) 
print(results_summary)

results_summary.to_csv('../results/model_performance_summary.csv', index=False)

    Dataset  Anatomy  Formal Logic  Moral Scenarios
0  Standard     0.92      0.979381             0.83
1       CoT     0.90      0.989247             0.84
2       RAG     0.00      0.000000             0.00


In [ ]:
#Generate a dataframe with 20 randomly selected errors for each model:
def get_error_samples(results_df, model_name):
    errors_df = results_df[results_df['correct'] == False]
    sample_errors_df = errors_df.sample(n=min(20, len(errors_df)), random_state=42)
    sample_errors_df['model'] = model_name
    return sample_errors_df

#Generate a csv file containing at most 20 randomly selected errors for each model
error_samples_std = get_error_samples(results_df_std, 'Standard')
error_samples_cot = get_error_samples(results_df_cot, 'CoT')
error_samples_rag = get_error_samples(results_df_rag, 'RAG')
error_samples_df = pd.concat([error_samples_std, error_samples_cot, error_samples_rag], ignore_index=True)
error_samples_df.to_csv('../results/model_error_samples.csv', index=False)